In [43]:
import torch
import torch.nn as nn
import torch.optim as optim

In [44]:
sequence = 'salom'
chars = sorted(list(set(sequence)))
print(chars)

['a', 'l', 'm', 'o', 's']


In [45]:
char2idx = {char: idx for idx, char in enumerate(chars)}
idx2char = {idx: char for idx, char in enumerate(chars)}

In [46]:
x_data = [char2idx[char] for char in sequence[:-1]] # 's', 'a', 'l', 'o'
y_data = [char2idx[char] for char in sequence[1:]] # 'a', 'l', 'o', 'm'

x = torch.tensor(x_data).unsqueeze(1) # (1, 4)
y = torch.tensor(y_data) # (4,)
x.shape, y.shape

(torch.Size([4, 1]), torch.Size([4]))

In [47]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(vocab_size, hidden_size)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, hidden):
        out, hidden = self.rnn(x, hidden)
        out = self.fc(out.squeeze(1))
        return out, hidden

In [48]:
vocab_size = len(chars)
hidden_size = 8
model = SimpleRNN(vocab_size, hidden_size)

In [49]:
def one_hot_encoding(index, vocab_size):
    vec = torch.zeros(1, 1, vocab_size)
    vec[0][0][index] = 1
    return vec

In [50]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [51]:
for epoch in range(100):
    loss = 0
    h = torch.zeros(1, 1, hidden_size)

    for i in range(len(x)):
        input = one_hot_encoding(x[i].item(), vocab_size)
        output, h = model(input, h.detach())
        loss += criterion(output, y[i].unsqueeze(0))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        pred_seq = ''
        h_test = torch.zeros(1, 1, hidden_size)

        for i in x_data:
            input = one_hot_encoding(i, vocab_size)
            output, h_test = model(input, h_test)
            pred_idx = output.argmax().item()
            pred_char = idx2char[pred_idx]
            pred_seq += pred_char
        print(f'Epoch {epoch}, Loss: {loss.item():.4f}, Predicted Sequence: {pred_seq}')

Epoch 0, Loss: 6.8619, Predicted Sequence: slss
Epoch 10, Loss: 4.9635, Predicted Sequence: alom
Epoch 20, Loss: 3.1364, Predicted Sequence: alom
Epoch 30, Loss: 1.5273, Predicted Sequence: alom
Epoch 40, Loss: 0.6601, Predicted Sequence: alom
Epoch 50, Loss: 0.3067, Predicted Sequence: alom
Epoch 60, Loss: 0.1724, Predicted Sequence: alom
Epoch 70, Loss: 0.1157, Predicted Sequence: alom
Epoch 80, Loss: 0.0871, Predicted Sequence: alom
Epoch 90, Loss: 0.0701, Predicted Sequence: alom
